# Week 1: Akan data prep and zero-shot Whisper baseline

Settings before running: **Accelerator → GPU T4 x2**, **Internet → On**, and attach your `fin-incl-akan` dataset (Add Input).

Run cells top to bottom. Each step can be re-run safely; finished audio conversions are skipped.

## 1. Get the code
Set `REPO_URL` to your GitHub repo. If the repo is private, make it public for now or add a GitHub token as a Kaggle secret.

In [ ]:
REPO_URL = "https://github.com/camycarl01/akan-asr.git"

!rm -rf /kaggle/working/akan-asr && git clone -q {REPO_URL} /kaggle/working/akan-asr
%cd /kaggle/working/akan-asr
!pip install -q -e ".[model]" 2>&1 | grep -v "already satisfied" | tail -3
!git log --oneline | head -3

## 2. Find the uploaded data
Kaggle puts attached datasets under `/kaggle/input/`. This finds each dialect folder by looking for its `data.csv` files, whatever the zip's nesting.

In [ ]:
from pathlib import Path

roots = {}
for csv in Path("/kaggle/input").rglob("data.csv"):
    folder = csv.parent.name.lower()          # e.g. fisd-asanti-twi-10p
    for dialect, key in [("asante", "asanti"), ("akuapem", "akuapim"), ("fante", "fanti")]:
        if key in folder:
            roots[dialect] = csv.parent.parent  # the folder holding 10p + 90p
for d in ["asante", "akuapem", "fante"]:
    print(f"{d:8s} -> {roots.get(d, 'NOT FOUND')}")
assert len(roots) == 3, "A dialect is missing: check the dataset is attached and unzipped"

## 3. Convert audio and build manifests
About 88,000 clips in total. Expect roughly 15–30 minutes with 4 CPU workers. Audio lands in `/kaggle/working/akan-asr/data/audio/`.

In [ ]:
for dialect, root in roots.items():
    !python scripts/prepare_data.py fin_incl --root "{root}" --dialect {dialect} --workers 4

## 4. Speaker-disjoint splits
Read both reports: hours/speakers per split, and how many test sentences were already seen in training (expect ~100% for this dataset, since everyone read the same prompts).

In [ ]:
!python scripts/make_splits.py data/manifests/fin_incl_*.csv

## 5. Smoke test the baseline (~2 minutes)
300 random test clips. Check the predictions look like text, not crashes or empty strings.

In [ ]:
!python scripts/run_baseline.py data/splits/fin_incl_test.csv --model openai/whisper-small --limit 300 --run-name smoke
import pandas as pd
pd.read_csv("results/smoke/fin_incl_predictions.csv")[["dialect", "reference", "prediction", "wer"]].sample(10, random_state=0)

## 6. Full zero-shot baseline
Your "before" numbers. Write these down: every later model is compared against them.

In [ ]:
!python scripts/run_baseline.py data/splits/fin_incl_test.csv --model openai/whisper-small

## 7. Save the small outputs
Manifests, splits and results are small CSV/JSON files. Download `week1_outputs.zip` from the Output panel, or save this notebook version ("Save & Run All") so Week 2 can attach its output.

In [ ]:
!zip -qr /kaggle/working/week1_outputs.zip data/manifests data/splits results && ls -lh /kaggle/working/week1_outputs.zip